In [1]:
# Cell 1: Install dependencies and Ollama
import subprocess
import time
import threading
import os

# First install zstd (required for Ollama extraction)
print("Installing zstd...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)

# Install Ollama
print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# Function to run Ollama server
def run_ollama_server():
    subprocess.run(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Start Ollama server in a separate thread (not background process)
print("Starting Ollama server...")
server_thread = threading.Thread(target=run_ollama_server, daemon=True)
server_thread.start()

# Wait for server to initialize
time.sleep(10)

# Pull the model (this will take several minutes)
print("Pulling llama3.1:8b (this takes 5-10 minutes)...")
subprocess.run(["ollama", "pull", "llama3.1:8b"], check=True)

print("✅ Ollama ready!")

Installing zstd...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,482 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,959 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu 

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 168 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (689 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Installing Ollama...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Starting Ollama server...
Pulling llama3.1:8b (this takes 5-10 minutes)...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  36 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   2% ▕                  ▏  88 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   4% ▕                  ▏ 182 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   6% ▕█                 ▏ 290 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   7% ▕█                 ▏ 338 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   9% ▕█                 ▏ 436 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  11% ▕█                 ▏ 537 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  12% ▕██                ▏ 580 MB/4.9 GB                  pulling manifes

✅ Ollama ready!


pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest ⠼ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest ⠴ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2

In [7]:
"""
extract_aspects_llm.py
======================
LLM-based narrative aspect extraction for SemEval-2026 Task 4.
Single combined prompt — all three aspects extracted in one LLM call.

Extracts three aspects per story, defined as follows (official task definitions):

  Course of Action (CoA)
    The SEQUENCE OF HAPPENINGS in the story — what happens, in what order,
    and what causes what. Described in abstract structural terms: what type
    of events occur and how they chain together. Character names and specific
    locations are replaced with role labels (protagonist, antagonist, ally).

  Outcomes
    The RESULTS OF THE HAPPENINGS, excluding intermediate results that change
    later on. Only the stable final state reached at the end of the story.
    Described in abstract terms: what the protagonist ultimately achieves,
    loses, or becomes.

  Abstract Theme
    The MOTIFS AND THEMES explored in the story. Does NOT cover the concrete
    setting. Universal human experiences, moral questions, and narrative
    archetypes — described without any reference to specific characters,
    locations, or plot events.

WHY THESE DEFINITIONS MATTER FOR SIMILARITY
  Two stories share a CoA when they have the same action structure:
    e.g. "criminal threatens protagonist → protagonist devises plan →
    plan executed → antagonist outwitted" matches across heist films.
  Two stories share an Outcome when they reach the same final state:
    e.g. "protagonist reunites with loved one after long separation"
    matches across very different plot paths.
  Two stories share a Theme when they explore the same human experiences:
    e.g. "loyalty vs institutional duty" matches regardless of genre.

PROMPT DESIGN PRINCIPLES
  1. One-shot example per aspect — shows the model the abstraction level
  2. Explicit negative constraints — forbid character names, specific places
  3. Role-based framing — ask for "protagonist" not character name
  4. Structured output — ask for numbered steps (CoA) or labelled sentences
  5. Length control — tight token budgets prevent over-generation

OUTPUT FORMAT
  aspects_cache_llm_extracted.json — same format as before:
  {"<story_text>": {"coa": "...", "outcomes": "...", "theme": "..."}, ...}
"""

import os, sys, json, time, random, re
from pathlib import Path
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_FILES = {
    "dev_track_a":  "dev_track_a.jsonl",
    "test_track_a": "test_track_a.jsonl",
    "test_track_b": "test_track_b.jsonl",
}

CACHE_PATH   = "/kaggle/working/aspects_cache_llm_extracted3.json"
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

# ══════════════════════════════════════════════════════════════════════════════
# COMBINED PROMPT
# Single prompt extracting all three aspects in one LLM call.
# Returns a JSON object with keys "coa", "outcomes", "theme".
#
# Design principles:
#   - All three aspect definitions in one context so the model sees
#     how they DIFFER from each other (critical for correct abstraction)
#   - One shared example showing all three aspects for the SAME story
#     (contrast between aspects is clearer than three separate examples)
#   - JSON output enforced via Ollama format parameter (valid JSON guaranteed)
#   - Explicit "having..." prohibition stops outcomes over-generation
#   - Role label requirement for CoA + Theme prevents name leakage
#   - Tight length guidance in output format section
# ══════════════════════════════════════════════════════════════════════════════

# System message — role and behavioural constraints
SYSTEM_MSG = (
    "You are a precise narrative analyst specialising in story structure. "
    "You follow instructions exactly. "
    "You always output valid JSON with no markdown fences, no extra keys, "
    "and no text outside the JSON object."
)

# Combined extraction prompt — one call, three aspects
COMBINED_PROMPT = """Analyse the story summary and extract three narrative aspects.
Return ONLY valid JSON with keys: "coa", "outcomes", "theme".
No extra text.

━━━ CORE PRINCIPLE ━━━
Each aspect must capture DIFFERENT information:
- COA = process (what happens)
- OUTCOMES = final state (what is true at the end)
- THEME = abstract meaning (what it represents)

Avoid overlap between them.

━━━━━━━━━━━━━━━━━━━━━━━
COA (Course of Action)
━━━━━━━━━━━━━━━━━━━━━━━
Describe the FULL causal sequence of events.

REQUIREMENTS:
- 3-6 numbered steps (1. 2. 3. ...)
- MUST include the FINAL transition into resolution
- Each step = one causal event
- Use abstract action types (escape, betrayal, investigation, confrontation, sacrifice)

USE:
- role labels (protagonist, antagonist, authority, ally)
- generic locations (city, prison, battlefield)

FORBIDDEN:
- character names
- specific places
- themes or emotions
- vague verbs ("deals with", "goes through")

GOOD:
"protagonist investigates → discovers threat → confronts antagonist → resolves conflict"

━━━━━━━━━━━━━━━━━━━━━━━
OUTCOMES (STRICT FORMAT)
━━━━━━━━━━━━━━━━━━━━━━━
Describe ONLY the final stable state.

Write EXACTLY 2 sentences:

Sentence 1:
- protagonist final status (success / failure / survival / transformation)

Sentence 2:
- type of resolution:
  - conflict_resolved / unresolved / partial
  - + nature of change (personal / relational / systemic)

FORBIDDEN:
- "having..." clauses
- process descriptions
- vague words like "things improve"

GOOD:
"Protagonist survives and achieves escape from immediate danger. The conflict is partially resolved with personal transformation but broader threats remain."

━━━━━━━━━━━━━━━━━━━━━━━
THEME (NORMALIZED)
━━━━━━━━━━━━━━━━━━━━━━━
Write 2-4 SHORT phrases (not full sentences).

Each phrase must be:
- abstract
- generalizable across stories

FORMAT:
"theme1; theme2; theme3"

GOOD:
"survival under adversity; loyalty vs duty; human resilience"

BAD:
"The story explores how a man struggles..."

━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE
━━━━━━━━━━━━━━━━━━━━━━━

Story:
"A young man injures his brother, is placed under supervision, falsely accused, and later proven innocent."

Output:
{
  "coa": "1. Protagonist commits violence and is processed by authority.\n2. Authority imposes supervision and assigns a helper.\n3. Community falsely accuses protagonist, escalating conflict.\n4. Evidence emerges that clears protagonist and resolves accusations.",
  "outcomes": "Protagonist is exonerated and transitions to a stable path of self-improvement. The conflict is resolved with personal transformation and social reintegration.",
  "theme": "redemption; social stigma; justice vs prejudice"
}

━━━━━━━━━━━━━━━━━━━━━━━
NOW ANALYSE
━━━━━━━━━━━━━━━━━━━━━━━

Story: {story}

Output:
"""

# Per-aspect validation thresholds
ASPECT_VALIDATORS = {
    "coa": {
        "min_len": 80,
        "max_len": 700,
        "warn_if_absent": r"^\s*1[\.\)]",
        "warn_msg": "CoA should start with numbered step '1.' or '1)'"
    },
    "outcomes": {
        "min_len": 40,
        "max_len": 300,
        "warn_if_absent": None,
        "warn_msg": ""
    },
    "theme": {
        "min_len": 60,
        "max_len": 450,
        "warn_if_absent": None,
        "warn_msg": ""
    },
}
# ══════════════════════════════════════════════════════════════════════════════
# STORY COLLECTION
# ══════════════════════════════════════════════════════════════════════════════

def _norm_key(text):
    return " ".join(str(text).split())


def collect_stories(data_files):
    seen, stories = set(), []

    def _add(text, title, source):
        if text is None:
            return
        key = _norm_key(text)
        if key and key not in seen and len(key) >= 20:
            seen.add(key)
            stories.append({"text": key, "title": title, "source": source})

    for name, path in data_files.items():
        if not Path(path).exists():
            print(f"  [skip] {path} not found")
            continue
        print(f"  Reading {path} ...")
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if name == "test_track_b":
                    _add(obj.get("text", ""),
                         obj.get("story_details", {}).get("title", ""), name)
                else:
                    for field, det_key in [
                        ("anchor_text", "story_details_anchor"),
                        ("text_a",      "story_details_a"),
                        ("text_b",      "story_details_b"),
                    ]:
                        _add(obj.get(field, ""),
                             obj.get(det_key, {}).get("title", ""), name)

    print(f"\nUnique stories collected: {len(stories)}")
    return stories


# ══════════════════════════════════════════════════════════════════════════════
# CACHE
# ══════════════════════════════════════════════════════════════════════════════

def load_cache(path):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        cache = {_norm_key(k): v for k, v in raw.items()}
        print(f"Cache loaded: {len(cache)} entries from {path}")
        return cache
    return {}


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def is_complete(entry):
    return (isinstance(entry, dict)
            and all(entry.get(k, "").strip() for k in ("coa", "outcomes", "theme")))



# ══════════════════════════════════════════════════════════════════════════════
# RESPONSE CLEANING & VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

# Strip preamble echoes from LLM responses (defence-in-depth after JSON parsing)
_PREAMBLE_RE = re.compile(
    r'^(?:'
    # "here is/are [the/a] <any phrase>" followed by newline
    r'here (?:is|are)(?: the| a)?[^\n:]{0,80}?[\s:.-]*\n+'
    r'|'
    # "Certainly!/Sure!" acknowledgements
    r'(?:certainly|sure|of course|absolutely)[!,.]?[^\n]*\n*'
    r'|'
    # Explicit aspect label ("Course of Action:", "Outcomes:", etc.)
    r'(?:course of action|outcomes?|abstract theme|response|answer)\s*[:：]\s*\n*'
    r')',
    re.IGNORECASE
)


def _clean_response(text):
    """Remove preamble echoes and leading labels from LLM responses."""
    if not text:
        return ""
    text = text.strip()
    for _ in range(4):
        cleaned = _PREAMBLE_RE.sub("", text).strip()
        if cleaned == text:
            break
        text = cleaned
    text = re.sub(r"^\s*\n+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _validate_aspect(aspect_name, text, story_preview):
    """Print a warning if an extracted aspect looks malformed."""
    v = ASPECT_VALIDATORS.get(aspect_name, {})
    if not v:
        return
    issues = []
    if len(text) < v.get("min_len", 0):
        issues.append(f"too short ({len(text)} chars, min={v['min_len']})")
    if len(text) > v.get("max_len", 9999):
        issues.append(f"very long ({len(text)} chars) — may be over-generated")
    pat = v.get("warn_if_absent")
    if pat and not re.search(pat, text, re.MULTILINE):
        issues.append(v.get("warn_msg", ""))
    if aspect_name in ("coa", "theme") and re.search(r'\([A-Z][a-z]+ [A-Z][a-z]+\)', text):
        issues.append("may contain actor name in parentheses")
    if issues:
        print(f"    \u26a0  {aspect_name}: {'; '.join(issues)}")
        print(f"       story: {story_preview[:60]}...")
        print(f"       text:  {text[:100]}")

# ══════════════════════════════════════════════════════════════════════════════
# OLLAMA BACKEND
# ══════════════════════════════════════════════════════════════════════════════

def _ollama_available():
    try:
        import urllib.request
        urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def check_ollama_gpu():
    import urllib.request
    try:
        with urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", "/api/ps"), timeout=5
        ) as resp:
            ps = json.loads(resp.read())
        for m in ps.get("models", []):
            vram = m.get("size_vram", 0)
            status = "GPU ✓" if vram > 0 else "CPU ⚠ slow!"
            print(f"  Ollama: {m.get('name')}  VRAM={vram/1e9:.1f}GB  {status}")
    except Exception:
        pass


def _ollama_generate(prompt, max_tokens=500, json_mode=False):
    """
    Call Ollama API.
    json_mode=True adds format="json" to the request — Ollama guarantees
    the response is valid JSON (llama3.1:8b supports this natively).
    """
    import urllib.request
    payload_dict = {
        "model":   OLLAMA_MODEL,
        "prompt":  prompt,
        "stream":  False,
        "options": {
            "num_predict":    max_tokens,
            "temperature":    0.1,   # low — factual extraction
            "top_p":          0.9,
            "repeat_penalty": 1.1,   # discourage prompt echo
        },
    }
    if json_mode:
        payload_dict["format"] = "json"

    payload = json.dumps(payload_dict).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read()).get("response", "").strip()
def _parse_json_response(raw):
    """
    Parse JSON from LLM response. Handles:
      - Clean JSON objects
      - JSON wrapped in markdown fences (```json ... ```)
      - Partial JSON with recoverable structure
    Returns dict with keys "coa", "outcomes", "theme", or raises ValueError.
    """
    text = raw.strip()

    # Strip markdown fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    # Attempt direct parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and all(k in obj for k in ("coa", "outcomes", "theme")):
            return obj
    except json.JSONDecodeError:
        pass

    # Recovery: extract individual fields with regex if JSON is malformed
    # (handles cases where the model emits valid field values but broken JSON syntax)
    recovered = {}
    for key in ("coa", "outcomes", "theme"):
        # Match: "key": "value" — handles escaped newlines and quotes
        pattern = rf'"{key}"\s*:\s*"((?:[^"\\]|\\.)*)"'
        m = re.search(pattern, text, re.DOTALL)
        if m:
            # Unescape the captured value
            recovered[key] = m.group(1).replace("\\n", "\n").replace('\\"', '"')

    if len(recovered) == 3:
        return recovered

    raise ValueError(f"Could not parse JSON response. Raw: {raw[:200]}")


def extract_with_ollama(story_text, title=""):
    """
    Extract all three narrative aspects in a single LLM call.
    Returns dict with keys: title, coa, outcomes, theme.
    """
    prompt   = SYSTEM_MSG + "\n\n" + COMBINED_PROMPT.format(story=story_text)
    aspects  = {"title": title}

    try:
        raw     = _ollama_generate(prompt, max_tokens=500, json_mode=True)
        parsed  = _parse_json_response(raw)
        aspects["coa"]      = _clean_response(parsed.get("coa",      ""))
        aspects["outcomes"] = _clean_response(parsed.get("outcomes", ""))
        aspects["theme"]    = _clean_response(parsed.get("theme",    ""))
    except Exception as e:
        print(f"    [ollama error] {e}")
        aspects["coa"] = aspects["outcomes"] = aspects["theme"] = ""
        return aspects

    # Validate each extracted aspect
    for asp in ("coa", "outcomes", "theme"):
        _validate_aspect(asp, aspects[asp], story_text)

    return aspects
# ══════════════════════════════════════════════════════════════════════════════
# CACHE REPAIR
# ══════════════════════════════════════════════════════════════════════════════

def repair_cache(cache_path):
    """
    Re-run _clean_response on all entries.
    Also flag entries where CoA is missing numbered steps (likely old-format).
    """
    cache = load_cache(cache_path)
    if not cache:
        print("Cache empty.")
        return
    cleaned_fields = 0
    suspicious = []
    for text, entry in cache.items():
        for asp in ("coa", "outcomes", "theme"):
            original = entry.get(asp, "")
            cleaned  = _clean_response(original)
            if cleaned != original:
                entry[asp] = cleaned
                cleaned_fields += 1
        # Flag CoA entries missing numbered structure (old extractions)
        coa = entry.get("coa", "")
        if coa and not re.search(r"^\s*1[\.\)]", coa, re.MULTILINE):
            suspicious.append((text[:60], coa[:80]))

    save_cache(cache, cache_path)
    print(f"Repair complete: {cleaned_fields} fields cleaned, "
          f"{len(suspicious)} CoA entries lack numbered steps.")
    if suspicious:
        print("First 3 suspicious CoA entries:")
        for story, coa in suspicious[:3]:
            print(f"  story: {story}...")
            print(f"  coa:   {coa}...")
    return cache


# ══════════════════════════════════════════════════════════════════════════════
# QUALITY CHECK
# ══════════════════════════════════════════════════════════════════════════════

def quality_check(cache, n=10):
    print("\n" + "=" * 70)
    print(f"  QUALITY CHECK — {min(n, len(cache))} random samples")
    print("=" * 70)

    for i, key in enumerate(random.sample(list(cache.keys()),
                                           min(n, len(cache))), 1):
        entry = cache[key]
        print(f"\n[{i}] {entry.get('title','(no title)')}")
        print(f"  Story : {key[:120]}...")
        print(f"  CoA   : {entry.get('coa','(empty)')[:200]}")
        print(f"  Out   : {entry.get('outcomes','(empty)')[:150]}")
        print(f"  Theme : {entry.get('theme','(empty)')[:150]}")
        for asp in ("coa", "outcomes", "theme"):
            _validate_aspect(asp, entry.get(asp, ""), key)

    total    = len(cache)
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"\n  Total: {total}  Complete: {complete} ({complete/total*100:.1f}%)"
          if total else "\n  Cache empty.")

    # Check CoA structural quality: fraction with numbered steps
    coa_structured = sum(
        1 for v in cache.values()
        if re.search(r"^\s*1[\.\)]", v.get("coa", ""), re.MULTILINE)
    )
    print(f"  CoA with numbered steps: {coa_structured}/{total} "
          f"({coa_structured/total*100:.1f}%)" if total else "")
    print("=" * 70 + "\n")


# ══════════════════════════════════════════════════════════════════════════════
# EXTRACTION LOOP
# ══════════════════════════════════════════════════════════════════════════════

def run_extraction(stories, cache, dry_run=False):
    todo = [s for s in stories if not is_complete(cache.get(s["text"]))]
    if dry_run:
        todo = todo[:5]
        print(f"DRY RUN: processing {len(todo)} stories only\n")
    else:
        print(f"To process: {len(todo)}  "
              f"Already cached: {len(stories) - len(todo)}\n")

    if not todo:
        print("Nothing to do — all stories cached.")
        return cache

    start = time.time()
    errors = 0

    for i, story in enumerate(tqdm(todo, desc="Extracting aspects")):
        try:
            aspects = extract_with_ollama(story["text"], story["title"])
            cache[story["text"]] = aspects
        except Exception as e:
            print(f"\n  [ERROR] story {i}: {e}")
            cache[story["text"]] = {
                "title": story["title"], "coa": "", "outcomes": "", "theme": ""}
            errors += 1

        # Save every 10 stories — crash recovery
        if (i + 1) % 10 == 0 or i == len(todo) - 1:
            save_cache(cache, CACHE_PATH)

        # Progress ETA every 50 stories
        if (i + 1) % 50 == 0:
            elapsed   = time.time() - start
            rate      = (i + 1) / elapsed
            remaining = (len(todo) - i - 1) / rate / 60
            print(f"\n  [{i+1}/{len(todo)}] "
                  f"{rate:.2f} stories/s  ETA {remaining:.1f} min  "
                  f"errors: {errors}")

    elapsed = time.time() - start
    print(f"\nDone in {elapsed/60:.1f} min  |  errors: {errors}")
    return cache


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG & MAIN
# ══════════════════════════════════════════════════════════════════════════════

class _Cfg:
    dry_run       = False   # True = process 5 stories only
    quality_check = False   # True = inspect cache and exit
    repair        = False   # True = clean existing cache entries and exit
    cache         = CACHE_PATH
    data_dir      = "/kaggle/input/datasets/anisoaraana/similarity"      # empty = use DATA_FILES paths as-is


def main():
    args = _Cfg()

    global CACHE_PATH
    CACHE_PATH = args.cache

    data_files = (
        {name: str(Path(args.data_dir) / fname)
         for name, fname in DATA_FILES.items()}
        if args.data_dir else dict(DATA_FILES)
    )

    print("=" * 60)
    print("  Narrative Aspect Extraction (LLM)")
    print("=" * 60)
    print(f"  Cache   : {CACHE_PATH}")
    print(f"  Backend : Ollama ({OLLAMA_MODEL})")
    print()

    cache = load_cache(CACHE_PATH)

    if args.repair:
        repair_cache(CACHE_PATH)
        return

    if args.quality_check:
        quality_check(cache) if cache else print("Cache empty.")
        return

    if not _ollama_available():
        print("ERROR: Ollama not running. Start with: ollama serve")
        print("       Then pull model: ollama pull llama3.1:8b")
        sys.exit(1)

    check_ollama_gpu()

    print("Collecting stories ...")
    stories = collect_stories(data_files)
    print()

    cache = run_extraction(stories, cache, dry_run=args.dry_run)

    if not args.dry_run:
        quality_check(cache, n=5)

    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"Final: {len(cache)} entries, {complete} complete")
    print(f"Cache: {CACHE_PATH}")


if __name__ == "__main__":
    main()

  Narrative Aspect Extraction (LLM)
  Cache   : /kaggle/working/aspects_cache_llm_extracted3.json
  Backend : Ollama (llama3.1:8b)

  Reading /kaggle/input/datasets/anisoaraana/similarity/dev_track_a.jsonl ...
  Reading /kaggle/input/datasets/anisoaraana/similarity/test_track_a.jsonl ...
  Reading /kaggle/input/datasets/anisoaraana/similarity/test_track_b.jsonl ...

Unique stories collected: 1688

To process: 1688  Already cached: 0



Extracting aspects:  22%|██▏       | 370/1688 [00:00<00:00, 3585.19it/s]


  [ERROR] story 0: '\n  "coa"'

  [ERROR] story 1: '\n  "coa"'

  [ERROR] story 2: '\n  "coa"'

  [ERROR] story 3: '\n  "coa"'

  [ERROR] story 4: '\n  "coa"'

  [ERROR] story 5: '\n  "coa"'

  [ERROR] story 6: '\n  "coa"'

  [ERROR] story 7: '\n  "coa"'

  [ERROR] story 8: '\n  "coa"'

  [ERROR] story 9: '\n  "coa"'

  [ERROR] story 10: '\n  "coa"'

  [ERROR] story 11: '\n  "coa"'

  [ERROR] story 12: '\n  "coa"'

  [ERROR] story 13: '\n  "coa"'

  [ERROR] story 14: '\n  "coa"'

  [ERROR] story 15: '\n  "coa"'

  [ERROR] story 16: '\n  "coa"'

  [ERROR] story 17: '\n  "coa"'

  [ERROR] story 18: '\n  "coa"'

  [ERROR] story 19: '\n  "coa"'

  [ERROR] story 20: '\n  "coa"'

  [ERROR] story 21: '\n  "coa"'

  [ERROR] story 22: '\n  "coa"'

  [ERROR] story 23: '\n  "coa"'

  [ERROR] story 24: '\n  "coa"'

  [ERROR] story 25: '\n  "coa"'

  [ERROR] story 26: '\n  "coa"'

  [ERROR] story 27: '\n  "coa"'

  [ERROR] story 28: '\n  "coa"'

  [ERROR] story 29: '\n  "coa"'

  [ERROR] story 30:

Extracting aspects:  43%|████▎     | 729/1688 [00:00<00:00, 1843.13it/s]


  [ERROR] story 710: '\n  "coa"'

  [ERROR] story 711: '\n  "coa"'

  [ERROR] story 712: '\n  "coa"'

  [ERROR] story 713: '\n  "coa"'

  [ERROR] story 714: '\n  "coa"'

  [ERROR] story 715: '\n  "coa"'

  [ERROR] story 716: '\n  "coa"'

  [ERROR] story 717: '\n  "coa"'

  [ERROR] story 718: '\n  "coa"'

  [ERROR] story 719: '\n  "coa"'

  [ERROR] story 720: '\n  "coa"'

  [ERROR] story 721: '\n  "coa"'

  [ERROR] story 722: '\n  "coa"'

  [ERROR] story 723: '\n  "coa"'

  [ERROR] story 724: '\n  "coa"'

  [ERROR] story 725: '\n  "coa"'

  [ERROR] story 726: '\n  "coa"'

  [ERROR] story 727: '\n  "coa"'

  [ERROR] story 728: '\n  "coa"'

  [ERROR] story 729: '\n  "coa"'

  [ERROR] story 730: '\n  "coa"'

  [ERROR] story 731: '\n  "coa"'

  [ERROR] story 732: '\n  "coa"'

  [ERROR] story 733: '\n  "coa"'

  [ERROR] story 734: '\n  "coa"'

  [ERROR] story 735: '\n  "coa"'

  [ERROR] story 736: '\n  "coa"'

  [ERROR] story 737: '\n  "coa"'

  [ERROR] story 738: '\n  "coa"'

  [ERROR] sto

...


  [ERROR] story 1570: '\n  "coa"'

  [ERROR] story 1571: '\n  "coa"'

  [ERROR] story 1572: '\n  "coa"'

  [ERROR] story 1573: '\n  "coa"'

  [ERROR] story 1574: '\n  "coa"'

  [ERROR] story 1575: '\n  "coa"'

  [ERROR] story 1576: '\n  "coa"'

  [ERROR] story 1577: '\n  "coa"'

  [ERROR] story 1578: '\n  "coa"'

  [ERROR] story 1579: '\n  "coa"'

  [ERROR] story 1580: '\n  "coa"'

  [ERROR] story 1581: '\n  "coa"'

  [ERROR] story 1582: '\n  "coa"'

  [ERROR] story 1583: '\n  "coa"'

  [ERROR] story 1584: '\n  "coa"'

  [ERROR] story 1585: '\n  "coa"'

  [ERROR] story 1586: '\n  "coa"'

  [ERROR] story 1587: '\n  "coa"'

  [ERROR] story 1588: '\n  "coa"'

  [ERROR] story 1589: '\n  "coa"'

  [ERROR] story 1590: '\n  "coa"'

  [ERROR] story 1591: '\n  "coa"'

  [ERROR] story 1592: '\n  "coa"'

  [ERROR] story 1593: '\n  "coa"'

  [ERROR] story 1594: '\n  "coa"'

  [ERROR] story 1595: '\n  "coa"'

  [ERROR] story 1596: '\n  "coa"'

  [ERROR] story 1597: '\n  "coa"'

  [ERROR] story 159

Extracting aspects:  97%|█████████▋| 1643/1688 [00:01<00:00, 585.05it/s]


  [ERROR] story 1640: '\n  "coa"'

  [ERROR] story 1641: '\n  "coa"'

  [ERROR] story 1642: '\n  "coa"'

  [ERROR] story 1643: '\n  "coa"'

  [ERROR] story 1644: '\n  "coa"'

  [ERROR] story 1645: '\n  "coa"'

  [ERROR] story 1646: '\n  "coa"'

  [ERROR] story 1647: '\n  "coa"'

  [ERROR] story 1648: '\n  "coa"'

  [ERROR] story 1649: '\n  "coa"'

  [1650/1688] 881.76 stories/s  ETA 0.0 min  errors: 1650

  [ERROR] story 1650: '\n  "coa"'

  [ERROR] story 1651: '\n  "coa"'

  [ERROR] story 1652: '\n  "coa"'

  [ERROR] story 1653: '\n  "coa"'

  [ERROR] story 1654: '\n  "coa"'

  [ERROR] story 1655: '\n  "coa"'

  [ERROR] story 1656: '\n  "coa"'

  [ERROR] story 1657: '\n  "coa"'

  [ERROR] story 1658: '\n  "coa"'

  [ERROR] story 1659: '\n  "coa"'

  [ERROR] story 1660: '\n  "coa"'

  [ERROR] story 1661: '\n  "coa"'

  [ERROR] story 1662: '\n  "coa"'

  [ERROR] story 1663: '\n  "coa"'

  [ERROR] story 1664: '\n  "coa"'

  [ERROR] story 1665: '\n  "coa"'

  [ERROR] story 1666: '\n  "co

Extracting aspects: 100%|██████████| 1688/1688 [00:01<00:00, 861.09it/s]


  [ERROR] story 1670: '\n  "coa"'

  [ERROR] story 1671: '\n  "coa"'

  [ERROR] story 1672: '\n  "coa"'

  [ERROR] story 1673: '\n  "coa"'

  [ERROR] story 1674: '\n  "coa"'

  [ERROR] story 1675: '\n  "coa"'

  [ERROR] story 1676: '\n  "coa"'

  [ERROR] story 1677: '\n  "coa"'

  [ERROR] story 1678: '\n  "coa"'

  [ERROR] story 1679: '\n  "coa"'

  [ERROR] story 1680: '\n  "coa"'

  [ERROR] story 1681: '\n  "coa"'

  [ERROR] story 1682: '\n  "coa"'

  [ERROR] story 1683: '\n  "coa"'

  [ERROR] story 1684: '\n  "coa"'

  [ERROR] story 1685: '\n  "coa"'

  [ERROR] story 1686: '\n  "coa"'

  [ERROR] story 1687: '\n  "coa"'

Done in 0.0 min  |  errors: 1688

  QUALITY CHECK — 5 random samples

[1] 
  Story : The wife of a rich man learns that her husband has an affair with a younger woman. She takes revenge on him by selling h...
  CoA   : 
  Out   : 
  Theme : 
    ⚠  coa: too short (0 chars, min=80); CoA should start with numbered step '1.' or '1)'
       story: The wife of a rich man 

In [8]:
"""
Aspect extraction

Extracts three narrative aspect descriptions per story:
  • Course of Action (CoA)  — what happens, in order
  • Outcomes               — the final resolution
  • Abstract Theme         — universal motifs and patterns

Output: aspects_cache.json
  {
    "<story_text>": {
        "title":    "...",
        "coa":      "...",
        "outcomes": "...",
        "theme":    "..."
    },
    ...
  }
"""

import os, sys, json, time, random, re
from pathlib import Path
from tqdm import tqdm

DATA_FILES = {
    "dev_track_a":  "dev_track_a.jsonl",    # 200 triples
    "test_track_a": "test_track_a.jsonl",   # 400 triples
    "test_track_b": "test_track_b.jsonl",   # 849 individual story texts
}

CACHE_PATH  = "/kaggle/working/aspects_cache.json"
OLLAMA_MODEL  = "llama3.1:8b"
OLLAMA_URL    = "http://localhost:11434/api/generate"

# Prompts

COA_PROMPT = """\
You are a narrative analyst. Read the story summary below and write ONLY \
the sequence of plot events — what happens, in what order, and what causes \
what. Do NOT mention character names, specific locations, or themes. \
Do NOT write any introduction, heading, or label before your answer. \
Begin your response immediately with the first event. Write 2-4 sentences.

Story:
{story}

Response:"""

OUTCOMES_PROMPT = """\
You are a narrative analyst. Read the story summary below and write ONLY \
the final outcome and resolution. What is the end state? What did the \
protagonist ultimately achieve, lose, or experience? Do NOT describe how \
they got there. Do NOT write any introduction, heading, or label. \
Begin your response immediately with the outcome. Write 1-2 sentences.

Story:
{story}

Response:"""

THEME_PROMPT = """\
You are a narrative analyst. Read the story summary below and write ONLY \
the abstract themes and universal human experiences it explores. What \
fundamental aspects of human nature, society, or morality does it examine? \
Do NOT mention specific characters, places, or plot events. \
Do NOT write any introduction, heading, or label. \
Begin your response immediately with the theme. Write 1-3 sentences.

Story:
{story}

Response:"""

PROMPTS = {
    "coa":      COA_PROMPT,
    "outcomes": OUTCOMES_PROMPT,
    "theme":    THEME_PROMPT,
}


# Story collection — deduplicate across all datasets

def collect_stories(data_files):
    """
    Walk all available dataset files and collect every unique story text.
    Returns a list of dicts: [{text, title, source}, ...]
    Deduplication is by exact text string after stripping whitespace.
    """
    seen   = set()
    stories = []

    def _add(text, title, source):
        # Guard against explicit JSON null values
        if text is None:
            return
        key = _normalise_key(text)   # same normalisation as cache keys
        if key and key not in seen and len(key) >= 20:
            seen.add(key)
            stories.append({"text": key, "title": title, "source": source})

    for name, path in data_files.items():
        if not Path(path).exists():
            print(f"  [skip] {path} not found")
            continue
        print(f"  Reading {path} ...")
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)

                if name == "test_track_b":
                    # Track B input: one story per row with "text" field
                    _add(obj.get("text",""),
                         obj.get("story_details",{}).get("title",""),
                         name)
                else:
                    # Track A / synthetic: triples with anchor_text, text_a, text_b
                    details = {
                        "anchor_text": obj.get("story_details_anchor",{}).get("title",""),
                        "text_a":      obj.get("story_details_a",{}).get("title",""),
                        "text_b":      obj.get("story_details_b",{}).get("title",""),
                    }
                    for field, title_key in [
                        ("anchor_text", "anchor_text"),
                        ("text_a",      "text_a"),
                        ("text_b",      "text_b"),
                    ]:
                        _add(obj.get(field,""), details[title_key], name)

    print(f"\nTotal unique stories collected: {len(stories)}")
    return stories


# Cache helpers

def _normalise_key(text):
    """Canonical cache key: strip whitespace, normalise internal spaces."""
    return " ".join(str(text).split())

def load_cache(path):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        # Re-key with normalised keys so laptop and Kaggle entries match
        cache = {_normalise_key(k): v for k, v in raw.items()}
        print(f"Cache loaded: {len(cache)} entries from {path}")
        return cache
    return {}


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def is_complete(entry):
    """An entry is complete if all three aspects are non-empty strings."""
    return (
        isinstance(entry, dict) and
        all(entry.get(k,"").strip() for k in ["coa", "outcomes", "theme"])
    )


# Backend: Ollama (local)

def _ollama_available():
    try:
        import urllib.request
        urllib.request.urlopen(OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def check_ollama_gpu():
    """
    Print Ollama GPU status. If Ollama is running on CPU instead of GPU,
    extraction will be ~10x slower. Run this before starting extraction.
    """
    import urllib.request
    try:
        with urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", "/api/ps"), timeout=5
        ) as resp:
            ps = json.loads(resp.read())
        models = ps.get("models", [])
        if models:
            for m in models:
                size_vram = m.get("size_vram", 0)
                print(f"  Ollama model: {m.get('name')}  "
                      f"VRAM: {size_vram/1e9:.1f}GB  "
                      f"({'GPU ✓' if size_vram > 0 else 'CPU ⚠ — slow!'})")
        else:
            print("  Ollama running but no model loaded yet (loads on first call)")
    except Exception:
        pass  # ps endpoint may not exist on older Ollama versions


def _ollama_generate(prompt, model=OLLAMA_MODEL, max_tokens=200):
    import urllib.request, urllib.error
    payload = json.dumps({
        "model":  model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": max_tokens,
            "temperature": 0.2,   # low temp for factual extraction
            "top_p": 0.9,
        }
    }).encode()

    req = urllib.request.Request(
        OLLAMA_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read())
    return result.get("response", "").strip()



# Response cleaning
# Llama-3.1-8B sometimes echoes a preamble like "Here is the sequence of
# plot events:" before the actual answer, despite explicit prompt instructions.
# This function removes all known preamble patterns and normalises whitespace.

_PREAMBLE_RE = re.compile(
    r'^(?:'
    r'(?:here (?:is|are) (?:the )?(?:a )?'
    r'(?:sequence of plot events|extracted plot events|following|'
    r'abstract themes?.*?|course of action.*?|outcome.*?|theme.*?))'
    r'[:\.!]?\s*\n+'
    r')',
    re.IGNORECASE | re.DOTALL
)

# Also strip leading label like "Course of Action:" or "Response:"
_LABEL_RE = re.compile(
    r'^(?:Course of Action|Outcome(?:s)?|Abstract Theme|Response)\s*[:：]\s*',
    re.IGNORECASE
)

def _clean_response(text):
    """Remove preamble echoes and leading labels from LLM responses."""
    text = text.strip()
    # Iteratively remove preamble lines (sometimes nested)
    for _ in range(3):
        cleaned = _PREAMBLE_RE.sub('', text).strip()
        if cleaned == text:
            break
        text = cleaned
    # Remove any remaining leading label
    text = _LABEL_RE.sub('', text).strip()
    # Normalise internal whitespace — collapse 3+ newlines to 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text


def extract_with_ollama(story_text, title=""):
    aspects = {}
    for aspect, prompt_template in PROMPTS.items():
        prompt = prompt_template.format(story=story_text)
        try:
            response = _ollama_generate(prompt)
            # Clean up: remove any accidental repetition of the prompt keyword
            response = _clean_response(response)
            aspects[aspect] = response
        except Exception as e:
            aspects[aspect] = ""
            print(f"    [ollama error] {aspect}: {e}")
    aspects["title"] = title
    return aspects


SYSTEM_MSG = (
    "You are a precise narrative analyst. "
    "Always follow instructions exactly. "
    "Never add introductions, headings, labels, or phrases like "
    "'Here is...' or 'Here are...' before your answer. "
    "Start your response immediately with the requested content."
)


def _build_messages(prompt):
    return [{"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": prompt}]


def _extract_reply(output):
    generated = output["generated_text"]
    if isinstance(generated, list):
        return generated[-1].get("content", "").strip()
    return str(generated).strip()

# Backend auto-detection and dispatch

def detect_backend(forced=None):
    if forced == "ollama":
        if not _ollama_available():
            print("ERROR: Ollama not running. Start it with: ollama serve")
            sys.exit(1)
        return "ollama"

    # Auto-detect: prefer Ollama if running locally
    if _ollama_available():
        print("Backend: Ollama (local) — detected running instance")
        return "ollama"


def extract_aspects(story_text, title, backend):
    if backend == "ollama":
        return extract_with_ollama(story_text, title)
    else:
        return 


# Quality check — print random samples from the cache

def quality_check(cache, n=10):
    print("\n" + "="*70)
    print(f"  QUALITY CHECK — {n} random samples from cache")
    print("="*70)

    sample_keys = random.sample(list(cache.keys()), min(n, len(cache)))
    for i, key in enumerate(sample_keys, 1):
        entry = cache[key]
        print(f"\n[{i}] Title: {entry.get('title', '(unknown)')}")
        print(f"    Story (first 120 chars): {key[:120]}...")
        print(f"    CoA:      {entry.get('coa','(empty)')[:150]}")
        print(f"    Outcomes: {entry.get('outcomes','(empty)')[:150]}")
        print(f"    Theme:    {entry.get('theme','(empty)')[:150]}")
        # Flag potential problems
        for asp in ["coa","outcomes","theme"]:
            val = entry.get(asp,"")
            if len(val) < 20:
                print(f"    ⚠  {asp} looks too short — may need manual fix")
            if any(w in val.lower() for w in ["romeo","juliet","hamlet"]):
                # proper names leaked into extraction
                print(f"    ⚠  {asp} may contain character names")

    print("\n" + "="*70)

    # Summary stats
    total = len(cache)
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"  Total entries : {total}")
    print(f"  Complete      : {complete}  ({complete/total*100:.1f}%)")
    print(f"  Incomplete    : {total-complete}")

    # Average lengths
    for asp in ["coa","outcomes","theme"]:
        lengths = [len(v.get(asp,"")) for v in cache.values() if v.get(asp,"")]
        if lengths:
            print(f"  Avg {asp:8s} length: {sum(lengths)/len(lengths):.0f} chars")
    print("="*70 + "\n")


# Main extraction loop

def run_extraction(stories, cache, backend, dry_run=False):
    # Filter to stories not yet in cache (or incomplete)
    todo = [s for s in stories if not is_complete(cache.get(s["text"]))]

    if dry_run:
        todo = todo[:5]
        print(f"DRY RUN: processing {len(todo)} stories only\n")
    else:
        print(f"Stories to process: {len(todo)} "
              f"(already cached: {len(stories)-len(todo)})\n")

    if not todo:
        print("Nothing to do — all stories already cached.")
        return cache

    start_time = time.time()
    errors = 0

    batch_size = 1

    for i in tqdm(range(0, len(todo), batch_size),
                  desc=f"Extracting ({backend})",
                  total=(len(todo) + batch_size - 1) // batch_size):

        batch = todo[i : i + batch_size]

        try:
            for story in batch:
                aspects = extract_aspects(story["text"], story["title"], backend)
                cache[story["text"]] = aspects
        except Exception as e:
            print(f"\n  [ERROR] batch {i}: {e}")
            for story in batch:
                cache[story["text"]] = {
                    "title": story["title"], "coa": "", "outcomes": "", "theme": ""}
            errors += len(batch)

        # Save every 10 batches — crash-safe
        batch_num = i // batch_size
        if batch_num % 10 == 0 or i + batch_size >= len(todo):
            save_cache(cache, CACHE_PATH)

        # Progress estimate every 50 stories
        stories_done = min(i + batch_size, len(todo))
        if stories_done > 0 and stories_done % 50 < batch_size:
            elapsed   = time.time() - start_time
            rate      = stories_done / elapsed
            remaining = (len(todo) - stories_done) / rate / 60
            print(f"\n  [{stories_done}/{len(todo)}] "
                  f"rate={rate:.2f} stories/s  "
                  f"ETA={remaining:.1f} min  "
                  f"errors={errors}")

    save_cache(cache, CACHE_PATH)
    elapsed = time.time() - start_time
    print(f"\nExtraction complete in {elapsed/60:.1f} min  |  errors: {errors}")
    return cache


# ── Cache repair — clean existing cache entries ───────────────────────────────

def repair_cache(cache_path):
    """
    Run _clean_response on every entry in an existing cache file.
    Use this to fix preamble echoes in caches generated with the old prompts.

    Usage:  python extract_aspects_kaggle.py  (set quality_check=False, 
            then call repair_cache(CACHE_PATH) directly)
    """
    cache = load_cache(cache_path)
    if not cache:
        print("Cache is empty or not found.")
        return

    fixed = 0
    for text, entry in cache.items():
        for asp in ["coa", "outcomes", "theme"]:
            original = entry.get(asp, "")
            cleaned  = _clean_response(original)
            if cleaned != original:
                entry[asp] = cleaned
                fixed += 1

    save_cache(cache, cache_path)
    print(f"Cache repair complete — {fixed} fields cleaned across {len(cache)} entries.")
    return cache


class _Cfg:

    backend       = "ollama" 

    # Set True to process only 5 stories — verify output before full run
    dry_run       = False

    # Set True to inspect existing cache and exit without extracting
    quality_check = False

    # ── Paths ─────────────────────────────────────────────────────────────
    # Cache output file
    cache         = CACHE_PATH
    data_dir      = "/kaggle/input/datasets/anisoaraana/similarity"

    hf_token      = ""   # leave empty

def parse_args():
    return _Cfg()


def main():
    args = parse_args()

    global CACHE_PATH
    CACHE_PATH = args.cache

    # Resolve data file paths
    # If data_dir is empty, DATA_FILES paths are used as-is (relative to cwd)
    if args.data_dir:
        data_files = {
            name: str(Path(args.data_dir) / fname)
            for name, fname in DATA_FILES.items()
        }
    else:
        data_files = dict(DATA_FILES)

    print("=" * 60)
    print("  Narrative Aspect Extraction")
    print("=" * 60)
    print(f"  Cache     : {CACHE_PATH}")
    print(f"  Data dir  : {args.data_dir}")
    print(f"  Backend   : {args.backend}")
    print()

    # Load existing cache
    cache = load_cache(CACHE_PATH)

    # Quality check mode — just inspect existing cache and exit
    if args.quality_check:
        if not cache:
            print("Cache is empty. Run extraction first.")
        else:
            quality_check(cache)
        return

    # Collect all unique stories from all available datasets
    print("Collecting stories from datasets...")
    stories = collect_stories(data_files)
    print()

    # Detect or validate backend
    backend = detect_backend(
        forced=args.backend if args.backend != "auto" else None
    )
    print(f"Using backend: {backend}")
    if backend == "ollama":
        check_ollama_gpu()
    print()

    # Run extraction
    cache = run_extraction(stories, cache, backend, dry_run=args.dry_run)

    # Always show quality check at end of a real (non-dry) run
    if not args.dry_run and cache:
        quality_check(cache, n=5)

    print(f"\nCache saved to: {CACHE_PATH}")
    print(f"Total entries : {len(cache)}")
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"Complete      : {complete}/{len(cache)}")


if __name__ == "__main__":
    main()

  Narrative Aspect Extraction
  Cache     : /kaggle/working/aspects_cache.json
  Data dir  : /kaggle/input/datasets/anisoaraana/similarity
  Backend   : ollama

  Reading /kaggle/input/datasets/anisoaraana/similarity/dev_track_a.jsonl ...
  Reading /kaggle/input/datasets/anisoaraana/similarity/test_track_a.jsonl ...
  Reading /kaggle/input/datasets/anisoaraana/similarity/test_track_b.jsonl ...

Total unique stories collected: 1688

Using backend: ollama
  Ollama running but no model loaded yet (loads on first call)

Stories to process: 1688 (already cached: 0)



Extracting (ollama):   1%|          | 11/1688 [01:40<4:14:05,  9.09s/it]


KeyboardInterrupt: 

In [2]:
"""
extract_aspects_llm.py
======================
LLM-based narrative aspect extraction for SemEval-2026 Task 4.
Single combined prompt — all three aspects extracted in one LLM call.

Extracts three aspects per story, defined as follows (official task definitions):

  Course of Action (CoA)
    The SEQUENCE OF HAPPENINGS in the story — what happens, in what order,
    and what causes what. Described in abstract structural terms: what type
    of events occur and how they chain together. Character names and specific
    locations are replaced with role labels (protagonist, antagonist, ally).

  Outcomes
    The RESULTS OF THE HAPPENINGS, excluding intermediate results that change
    later on. Only the stable final state reached at the end of the story.
    Described in abstract terms: what the protagonist ultimately achieves,
    loses, or becomes.

  Abstract Theme
    The MOTIFS AND THEMES explored in the story. Does NOT cover the concrete
    setting. Universal human experiences, moral questions, and narrative
    archetypes — described without any reference to specific characters,
    locations, or plot events.

WHY THESE DEFINITIONS MATTER FOR SIMILARITY
  Two stories share a CoA when they have the same action structure:
    e.g. "criminal threatens protagonist → protagonist devises plan →
    plan executed → antagonist outwitted" matches across heist films.
  Two stories share an Outcome when they reach the same final state:
    e.g. "protagonist reunites with loved one after long separation"
    matches across very different plot paths.
  Two stories share a Theme when they explore the same human experiences:
    e.g. "loyalty vs institutional duty" matches regardless of genre.

PROMPT DESIGN PRINCIPLES
  1. One-shot example per aspect — shows the model the abstraction level
  2. Explicit negative constraints — forbid character names, specific places
  3. Role-based framing — ask for "protagonist" not character name
  4. Structured output — ask for numbered steps (CoA) or labelled sentences
  5. Length control — tight token budgets prevent over-generation

OUTPUT FORMAT
  aspects_cache_llm_extracted.json — same format as before:
  {"<story_text>": {"coa": "...", "outcomes": "...", "theme": "..."}, ...}
"""

import os, sys, json, time, random, re
from pathlib import Path
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_FILES = {
    "dev_track_a":  "dev_track_a.jsonl",
    "test_track_a": "test_track_a.jsonl",
    "test_track_b": "test_track_b.jsonl",
}

CACHE_PATH   = "/kaggle/working/aspects_cache_llm_extracted5.json"
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

# ══════════════════════════════════════════════════════════════════════════════
# COMBINED PROMPT
# Single prompt extracting all three aspects in one LLM call.
# Returns a JSON object with keys "coa", "outcomes", "theme".
#
# Design principles:
#   - All three aspect definitions in one context so the model sees
#     how they DIFFER from each other (critical for correct abstraction)
#   - One shared example showing all three aspects for the SAME story
#     (contrast between aspects is clearer than three separate examples)
#   - JSON output enforced via Ollama format parameter (valid JSON guaranteed)
#   - Explicit "having..." prohibition stops outcomes over-generation
#   - Role label requirement for CoA + Theme prevents name leakage
#   - Tight length guidance in output format section
# ══════════════════════════════════════════════════════════════════════════════

# System message — role and behavioural constraints
SYSTEM_MSG = (
    "You are a precise narrative analyst specialising in story structure. "
    "You follow instructions exactly. "
    "You always output valid JSON with no markdown fences, no extra keys, "
    "and no text outside the JSON object."
)

# Combined extraction prompt — one call, three aspects
COMBINED_PROMPT = """Analyse the story summary and extract three narrative aspects.
Return ONLY valid JSON with keys: "coa", "outcomes", "theme".
No extra text.

━━━ CORE PRINCIPLE ━━━
Each aspect must capture DIFFERENT information:
- COA = process (what happens)
- OUTCOMES = final state (what is true at the end)
- THEME = abstract meaning (what it represents)

Avoid overlap between them.

━━━━━━━━━━━━━━━━━━━━━━━
COA (Course of Action)
━━━━━━━━━━━━━━━━━━━━━━━
Describe the FULL causal sequence of events.

REQUIREMENTS:
- 3-6 numbered steps (1. 2. 3. ...)
- MUST include the FINAL transition into resolution
- Each step = one causal event
- Use abstract action types (escape, betrayal, investigation, confrontation, sacrifice)

USE:
- role labels (protagonist, antagonist, authority, ally)
- generic locations (city, prison, battlefield)

FORBIDDEN:
- character names
- specific places
- themes or emotions
- vague verbs ("deals with", "goes through")

GOOD:
"protagonist investigates → discovers threat → confronts antagonist → resolves conflict"

━━━━━━━━━━━━━━━━━━━━━━━
OUTCOMES (STRICT FORMAT)
━━━━━━━━━━━━━━━━━━━━━━━
Describe ONLY the final stable state.

Write EXACTLY 2 sentences:

Sentence 1:
- protagonist final status (success / failure / survival / transformation)

Sentence 2:
- type of resolution:
  - conflict_resolved / unresolved / partial
  - + nature of change (personal / relational / systemic)

FORBIDDEN:
- "having..." clauses
- process descriptions
- vague words like "things improve"

GOOD:
"Protagonist survives and achieves escape from immediate danger. The conflict is partially resolved with personal transformation but broader threats remain."

━━━━━━━━━━━━━━━━━━━━━━━
THEME (NORMALIZED)
━━━━━━━━━━━━━━━━━━━━━━━
Write 2-4 SHORT phrases (not full sentences).

Each phrase must be:
- abstract
- generalizable across stories

FORMAT:
"theme1; theme2; theme3"

GOOD:
"survival under adversity; loyalty vs duty; human resilience"

BAD:
"The story explores how a man struggles..."

━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE
━━━━━━━━━━━━━━━━━━━━━━━

Story:
"A young man injures his brother, is placed under supervision, falsely accused, and later proven innocent."

Output:
{{
  "coa": "1. Protagonist commits violence and is processed by authority.\n2. Authority imposes supervision and assigns a helper.\n3. Community falsely accuses protagonist, escalating conflict.\n4. Evidence emerges that clears protagonist and resolves accusations.",
  "outcomes": "Protagonist is exonerated and transitions to a stable path of self-improvement. The conflict is resolved with personal transformation and social reintegration.",
  "theme": "redemption; social stigma; justice vs prejudice"
}}

━━━━━━━━━━━━━━━━━━━━━━━
NOW ANALYSE
━━━━━━━━━━━━━━━━━━━━━━━

Story: {story}

Output:
"""

# Per-aspect validation thresholds
ASPECT_VALIDATORS = {
    "coa": {
        "min_len": 80,
        "max_len": 700,
        "warn_if_absent": r"^\s*1[\.\)]",
        "warn_msg": "CoA should start with numbered step '1.' or '1)'"
    },
    "outcomes": {
        "min_len": 40,
        "max_len": 300,
        "warn_if_absent": None,
        "warn_msg": ""
    },
    "theme": {
        "min_len": 60,
        "max_len": 450,
        "warn_if_absent": None,
        "warn_msg": ""
    },
}
# ══════════════════════════════════════════════════════════════════════════════
# STORY COLLECTION
# ══════════════════════════════════════════════════════════════════════════════

def _norm_key(text):
    return " ".join(str(text).split())


def collect_stories(data_files):
    seen, stories = set(), []

    def _add(text, title, source):
        if text is None:
            return
        key = _norm_key(text)
        if key and key not in seen and len(key) >= 20:
            seen.add(key)
            stories.append({"text": key, "title": title, "source": source})

    for name, path in data_files.items():
        if not Path(path).exists():
            print(f"  [skip] {path} not found")
            continue
        print(f"  Reading {path} ...")
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if name == "test_track_b":
                    _add(obj.get("text", ""),
                         obj.get("story_details", {}).get("title", ""), name)
                else:
                    for field, det_key in [
                        ("anchor_text", "story_details_anchor"),
                        ("text_a",      "story_details_a"),
                        ("text_b",      "story_details_b"),
                    ]:
                        _add(obj.get(field, ""),
                             obj.get(det_key, {}).get("title", ""), name)

    print(f"\nUnique stories collected: {len(stories)}")
    return stories


# ══════════════════════════════════════════════════════════════════════════════
# CACHE
# ══════════════════════════════════════════════════════════════════════════════

def load_cache(path):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        cache = {_norm_key(k): v for k, v in raw.items()}
        print(f"Cache loaded: {len(cache)} entries from {path}")
        return cache
    return {}


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def is_complete(entry):
    return (isinstance(entry, dict)
            and all(entry.get(k, "").strip() for k in ("coa", "outcomes", "theme")))



# ══════════════════════════════════════════════════════════════════════════════
# RESPONSE CLEANING & VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

# Strip preamble echoes from LLM responses (defence-in-depth after JSON parsing)
_PREAMBLE_RE = re.compile(
    r'^(?:'
    # "here is/are [the/a] <any phrase>" followed by newline
    r'here (?:is|are)(?: the| a)?[^\n:]{0,80}?[\s:.-]*\n+'
    r'|'
    # "Certainly!/Sure!" acknowledgements
    r'(?:certainly|sure|of course|absolutely)[!,.]?[^\n]*\n*'
    r'|'
    # Explicit aspect label ("Course of Action:", "Outcomes:", etc.)
    r'(?:course of action|outcomes?|abstract theme|response|answer)\s*[:：]\s*\n*'
    r')',
    re.IGNORECASE
)


def _clean_response(text):
    """Remove preamble echoes and leading labels from LLM responses."""
    if not text:
        return ""
    text = text.strip()
    for _ in range(4):
        cleaned = _PREAMBLE_RE.sub("", text).strip()
        if cleaned == text:
            break
        text = cleaned
    text = re.sub(r"^\s*\n+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _validate_aspect(aspect_name, text, story_preview):
    """Print a warning if an extracted aspect looks malformed."""
    v = ASPECT_VALIDATORS.get(aspect_name, {})
    if not v:
        return
    issues = []
    if len(text) < v.get("min_len", 0):
        issues.append(f"too short ({len(text)} chars, min={v['min_len']})")
    if len(text) > v.get("max_len", 9999):
        issues.append(f"very long ({len(text)} chars) — may be over-generated")
    pat = v.get("warn_if_absent")
    if pat and not re.search(pat, text, re.MULTILINE):
        issues.append(v.get("warn_msg", ""))
    if aspect_name in ("coa", "theme") and re.search(r'\([A-Z][a-z]+ [A-Z][a-z]+\)', text):
        issues.append("may contain actor name in parentheses")
    if issues:
        print(f"    \u26a0  {aspect_name}: {'; '.join(issues)}")
        print(f"       story: {story_preview[:60]}...")
        print(f"       text:  {text[:100]}")

# ══════════════════════════════════════════════════════════════════════════════
# OLLAMA BACKEND
# ══════════════════════════════════════════════════════════════════════════════

def _ollama_available():
    try:
        import urllib.request
        urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def check_ollama_gpu():
    import urllib.request
    try:
        with urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", "/api/ps"), timeout=5
        ) as resp:
            ps = json.loads(resp.read())
        for m in ps.get("models", []):
            vram = m.get("size_vram", 0)
            status = "GPU ✓" if vram > 0 else "CPU ⚠ slow!"
            print(f"  Ollama: {m.get('name')}  VRAM={vram/1e9:.1f}GB  {status}")
    except Exception:
        pass


def _ollama_generate(prompt, max_tokens=500, json_mode=False):
    """
    Call Ollama API.
    json_mode=True adds format="json" to the request — Ollama guarantees
    the response is valid JSON (llama3.1:8b supports this natively).
    """
    import urllib.request
    payload_dict = {
        "model":   OLLAMA_MODEL,
        "prompt":  prompt,
        "stream":  False,
        "options": {
            "num_predict":    max_tokens,
            "temperature":    0.1,   # low — factual extraction
            "top_p":          0.9,
            "repeat_penalty": 1.1,   # discourage prompt echo
        },
    }
    if json_mode:
        payload_dict["format"] = "json"

    payload = json.dumps(payload_dict).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read()).get("response", "").strip()
def _parse_json_response(raw):
    """
    Parse JSON from LLM response. Handles:
      - Clean JSON objects
      - JSON wrapped in markdown fences (```json ... ```)
      - Partial JSON with recoverable structure
    Returns dict with keys "coa", "outcomes", "theme", or raises ValueError.
    """
    text = raw.strip()

    # Strip markdown fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    # Attempt direct parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and all(k in obj for k in ("coa", "outcomes", "theme")):
            return obj
    except json.JSONDecodeError:
        pass

    # Recovery: extract individual fields with regex if JSON is malformed
    # (handles cases where the model emits valid field values but broken JSON syntax)
    recovered = {}
    for key in ("coa", "outcomes", "theme"):
        # Match: "key": "value" — handles escaped newlines and quotes
        pattern = rf'"{key}"\s*:\s*"((?:[^"\\]|\\.)*)"'
        m = re.search(pattern, text, re.DOTALL)
        if m:
            # Unescape the captured value
            recovered[key] = m.group(1).replace("\\n", "\n").replace('\\"', '"')

    if len(recovered) == 3:
        return recovered

    raise ValueError(f"Could not parse JSON response. Raw: {raw[:200]}")


def extract_with_ollama(story_text, title=""):
    """
    Extract all three narrative aspects in a single LLM call.
    Returns dict with keys: title, coa, outcomes, theme.
    """
    prompt   = SYSTEM_MSG + "\n\n" + COMBINED_PROMPT.format(story=story_text)
    aspects  = {"title": title}

    try:
        raw     = _ollama_generate(prompt, max_tokens=500, json_mode=True)
        parsed  = _parse_json_response(raw)
        aspects["coa"]      = _clean_response(parsed.get("coa",      ""))
        aspects["outcomes"] = _clean_response(parsed.get("outcomes", ""))
        aspects["theme"]    = _clean_response(parsed.get("theme",    ""))
    except Exception as e:
        print(f"    [ollama error] {e}")
        aspects["coa"] = aspects["outcomes"] = aspects["theme"] = ""
        return aspects

    # Validate each extracted aspect
    for asp in ("coa", "outcomes", "theme"):
        _validate_aspect(asp, aspects[asp], story_text)

    return aspects
# ══════════════════════════════════════════════════════════════════════════════
# CACHE REPAIR
# ══════════════════════════════════════════════════════════════════════════════

def repair_cache(cache_path):
    """
    Re-run _clean_response on all entries.
    Also flag entries where CoA is missing numbered steps (likely old-format).
    """
    cache = load_cache(cache_path)
    if not cache:
        print("Cache empty.")
        return
    cleaned_fields = 0
    suspicious = []
    for text, entry in cache.items():
        for asp in ("coa", "outcomes", "theme"):
            original = entry.get(asp, "")
            cleaned  = _clean_response(original)
            if cleaned != original:
                entry[asp] = cleaned
                cleaned_fields += 1
        # Flag CoA entries missing numbered structure (old extractions)
        coa = entry.get("coa", "")
        if coa and not re.search(r"^\s*1[\.\)]", coa, re.MULTILINE):
            suspicious.append((text[:60], coa[:80]))

    save_cache(cache, cache_path)
    print(f"Repair complete: {cleaned_fields} fields cleaned, "
          f"{len(suspicious)} CoA entries lack numbered steps.")
    if suspicious:
        print("First 3 suspicious CoA entries:")
        for story, coa in suspicious[:3]:
            print(f"  story: {story}...")
            print(f"  coa:   {coa}...")
    return cache


# ══════════════════════════════════════════════════════════════════════════════
# QUALITY CHECK
# ══════════════════════════════════════════════════════════════════════════════

def quality_check(cache, n=10):
    print("\n" + "=" * 70)
    print(f"  QUALITY CHECK — {min(n, len(cache))} random samples")
    print("=" * 70)

    for i, key in enumerate(random.sample(list(cache.keys()),
                                           min(n, len(cache))), 1):
        entry = cache[key]
        print(f"\n[{i}] {entry.get('title','(no title)')}")
        print(f"  Story : {key[:120]}...")
        print(f"  CoA   : {entry.get('coa','(empty)')[:200]}")
        print(f"  Out   : {entry.get('outcomes','(empty)')[:150]}")
        print(f"  Theme : {entry.get('theme','(empty)')[:150]}")
        for asp in ("coa", "outcomes", "theme"):
            _validate_aspect(asp, entry.get(asp, ""), key)

    total    = len(cache)
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"\n  Total: {total}  Complete: {complete} ({complete/total*100:.1f}%)"
          if total else "\n  Cache empty.")

    # Check CoA structural quality: fraction with numbered steps
    coa_structured = sum(
        1 for v in cache.values()
        if re.search(r"^\s*1[\.\)]", v.get("coa", ""), re.MULTILINE)
    )
    print(f"  CoA with numbered steps: {coa_structured}/{total} "
          f"({coa_structured/total*100:.1f}%)" if total else "")
    print("=" * 70 + "\n")


# ══════════════════════════════════════════════════════════════════════════════
# EXTRACTION LOOP
# ══════════════════════════════════════════════════════════════════════════════

def run_extraction(stories, cache, dry_run=False):
    todo = [s for s in stories if not is_complete(cache.get(s["text"]))]
    if dry_run:
        todo = todo[:5]
        print(f"DRY RUN: processing {len(todo)} stories only\n")
    else:
        print(f"To process: {len(todo)}  "
              f"Already cached: {len(stories) - len(todo)}\n")

    if not todo:
        print("Nothing to do — all stories cached.")
        return cache

    start = time.time()
    errors = 0

    for i, story in enumerate(tqdm(todo, desc="Extracting aspects")):
        try:
            aspects = extract_with_ollama(story["text"], story["title"])
            cache[story["text"]] = aspects
        except Exception as e:
            print(f"\n  [ERROR] story {i}: {e}")
            cache[story["text"]] = {
                "title": story["title"], "coa": "", "outcomes": "", "theme": ""}
            errors += 1

        # Save every 10 stories — crash recovery
        if (i + 1) % 10 == 0 or i == len(todo) - 1:
            save_cache(cache, CACHE_PATH)

        # Progress ETA every 50 stories
        if (i + 1) % 50 == 0:
            elapsed   = time.time() - start
            rate      = (i + 1) / elapsed
            remaining = (len(todo) - i - 1) / rate / 60
            print(f"\n  [{i+1}/{len(todo)}] "
                  f"{rate:.2f} stories/s  ETA {remaining:.1f} min  "
                  f"errors: {errors}")

    elapsed = time.time() - start
    print(f"\nDone in {elapsed/60:.1f} min  |  errors: {errors}")
    return cache


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG & MAIN
# ══════════════════════════════════════════════════════════════════════════════

class _Cfg:
    dry_run       = False   # True = process 5 stories only
    quality_check = False   # True = inspect cache and exit
    repair        = False   # True = clean existing cache entries and exit
    cache         = CACHE_PATH
    data_dir      = "/kaggle/input/datasets/anisoaraana/similarity"      # empty = use DATA_FILES paths as-is


def main():
    args = _Cfg()

    global CACHE_PATH
    CACHE_PATH = args.cache

    data_files = (
        {name: str(Path(args.data_dir) / fname)
         for name, fname in DATA_FILES.items()}
        if args.data_dir else dict(DATA_FILES)
    )

    print("=" * 60)
    print("  Narrative Aspect Extraction (LLM)")
    print("=" * 60)
    print(f"  Cache   : {CACHE_PATH}")
    print(f"  Backend : Ollama ({OLLAMA_MODEL})")
    print()

    cache = load_cache(CACHE_PATH)

    if args.repair:
        repair_cache(CACHE_PATH)
        return

    if args.quality_check:
        quality_check(cache) if cache else print("Cache empty.")
        return

    if not _ollama_available():
        print("ERROR: Ollama not running. Start with: ollama serve")
        print("       Then pull model: ollama pull llama3.1:8b")
        sys.exit(1)

    check_ollama_gpu()

    print("Collecting stories ...")
    stories = collect_stories(data_files)
    print()

    cache = run_extraction(stories, cache, dry_run=args.dry_run)

    if not args.dry_run:
        quality_check(cache, n=5)

    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"Final: {len(cache)} entries, {complete} complete")
    print(f"Cache: {CACHE_PATH}")


if __name__ == "__main__":
    main()


  Narrative Aspect Extraction (LLM)
  Cache   : /kaggle/working/aspects_cache_llm_extracted5.json
  Backend : Ollama (llama3.1:8b)

Cache loaded: 490 entries from /kaggle/working/aspects_cache_llm_extracted5.json
  Reading /kaggle/input/datasets/anisoaraana/similarity/dev_track_a.jsonl ...
  Reading /kaggle/input/datasets/anisoaraana/similarity/test_track_a.jsonl ...
  Reading /kaggle/input/datasets/anisoaraana/similarity/test_track_b.jsonl ...

Unique stories collected: 1688

To process: 1198  Already cached: 490



Extracting aspects:   0%|          | 1/1198 [01:25<28:34:40, 85.95s/it]

    ⚠  coa: may contain actor name in parentheses
       story: Retired cardiologist Ben Givens lives in Seattle and has rec...
       text:  1. Protagonist (Ben Givens) decides to end his life due to cancer diagnosis → informs daughter Renee
    ⚠  theme: too short (45 chars, min=60)
       story: Retired cardiologist Ben Givens lives in Seattle and has rec...
       text:  suicide; loss and grief; memory and nostalgia


Extracting aspects:   0%|          | 2/1198 [01:33<13:13:00, 39.78s/it]

    ⚠  theme: too short (38 chars, min=60)
       story: Janet, a divorced, middle-aged writer who has become somewha...
       text:  acceptance; changing facts; letting go


...

    ⚠  coa: may contain actor name in parentheses
       story: The reclusive, eccentric scientist Oscar Collins (Jack MacGo...
       text:  1. Protagonist (Oscar Collins) discovers a beam of light streaming through a hole in the wall betwee
    ⚠  theme: too short (50 chars, min=60)
       story: The reclusive, eccentric scientist Oscar Collins (Jack MacGo...
       text:  obsession; invasion of privacy; blurred boundaries


Extracting aspects: 100%|█████████▉| 1197/1198 [2:40:20<00:07,  8.00s/it]

    ⚠  theme: too short (48 chars, min=60)
       story: In most versions, the eponymous hero is hunted and persecute...
       text:  punishment of lust; rewards of love and fidelity


Extracting aspects: 100%|██████████| 1198/1198 [2:40:29<00:00,  8.04s/it]

    ⚠  theme: too short (45 chars, min=60)
       story: Upon his return to Italy from his many adventures, the great...
       text:  redemption; loyalty vs duty; human resilience

Done in 160.5 min  |  errors: 0

  QUALITY CHECK — 5 random samples

[1] 
  Story : Forced to leave France, Dédée and her bullying pimp Marco have reached Antwerp, where she is one of the girls in René's ...
  CoA   : 1. Protagonist Dédée and antagonist Marco relocate from France to Antwerp. 2. Dédée meets Francesco, who offers her escape with his cargo ship. 3. René agrees to facilitate their departure in exchange
  Out   : Protagonist Dédée achieves escape from her abusive situation with the help of ally René. The conflict is resolved with Marco's death, but broader thre
  Theme : escape from oppression; loyalty vs duty; human resilience
    ⚠  theme: too short (57 chars, min=60)
       story: Forced to leave France, Dédée and her bullying pimp Marco ha...
       text:  escape from oppression; loyalty vs 